In [8]:
import pandas as pd
import numpy as np
from dotenv import load_dotenv
import os

In [9]:
load_dotenv()
BASE_PATH = os.getenv("BASE_PATH")
pd.set_option('display.max_columns', None)
df = pd.read_csv(f"{BASE_PATH}/data/raw/weather_training_data.csv")
df.head()

,row ID,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,WindDir3pm,WindSpeed9am,WindSpeed3pm,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow
0,Row0,Albury,13.4,22.9,0.6,NaN,NaN,W,44.0,W,WNW,20.0,24.0,71.0,22.0,1007.7,1007.1,8.0,NaN,16.9,21.8,No,0
1,Row1,Albury,7.4,25.1,0.0,NaN,NaN,WNW,44.0,NNW,WSW,4.0,22.0,44.0,25.0,1010.6,1007.8,NaN,NaN,17.2,24.3,No,0
2,Row2,Albury,17.5,32.3,1.0,NaN,NaN,W,41.0,ENE,NW,7.0,20.0,82.0,33.0,1010.8,1006.0,7.0,8.0,17.8,29.7,No,0
3,Row3,Albury,14.6,29.7,0.2,NaN,NaN,WNW,56.0,W,W,19.0,24.0,55.0,23.0,1009.2,1005.4,NaN,NaN,20.6,28.9,No,0
4,Row4,Albury,7.7,26.7,0.0,NaN,NaN,W,35.0,SSE,W,6.0,17.0,48.0,19.0,1013.4,1010.1,NaN,NaN,16.3,25.5,No,0


In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 99516 entries, 0 to 99515
Data columns (total 23 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   row ID         99516 non-null  str    
 1   Location       99516 non-null  str    
 2   MinTemp        99073 non-null  float64
 3   MaxTemp        99286 non-null  float64
 4   Rainfall       98537 non-null  float64
 5   Evaporation    56985 non-null  float64
 6   Sunshine       52199 non-null  float64
 7   WindGustDir    92995 non-null  str    
 8   WindGustSpeed  93036 non-null  float64
 9   WindDir9am     92510 non-null  str    
 10  WindDir3pm     96868 non-null  str    
 11  WindSpeed9am   98581 non-null  float64
 12  WindSpeed3pm   97681 non-null  float64
 13  Humidity9am    98283 non-null  float64
 14  Humidity3pm    97010 non-null  float64
 15  Pressure9am    89768 non-null  float64
 16  Pressure3pm    89780 non-null  float64
 17  Cloud9am       61944 non-null  float64
 18  Cloud3pm       59

In [ ]:
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv

# Load Data 
load_dotenv()
BASE_PATH = os.getenv("BASE_PATH")
OUTPUT_PATH = f"{BASE_PATH}/data/processed/clean_weather_training_data.csv"
df = pd.read_csv(f"{BASE_PATH}/data/raw/weather_training_data.csv")

# Drop Identifier Column 
df.drop(columns=['row ID'], inplace=True)


# Drop Duplicates
df = df.drop_duplicates()

# Range of Cloud According to Oktas from 0 to 8
df.loc[df['Cloud3pm'] == 9.0, 'Cloud3pm'] = np.nan

# Filling Missing Values With Median
numerical_columns = ['MaxTemp', 'MinTemp', 'Rainfall', 'WindGustSpeed', 'WindSpeed9am', 'WindSpeed3pm', 'Humidity3pm', 'Humidity9am', 'Pressure9am', 'Pressure3pm', 'Temp9am', 'Temp3pm']
for col in numerical_columns: 
    df[col] = df.groupby('Location')[col].transform(
    lambda x: x.fillna(x.median())
)
    df[col] = df[col].fillna(df[col].median())   # global fallback


# Filling Wind Directions With Mode 
df['WindGustDir'] = df.groupby('Location')['WindGustDir'].transform(
    lambda x: x.fillna(x.mode()[0] if not x.mode().empty else x)
)

global_mode = df['WindGustDir'].mode()[0]
df['WindGustDir'] = df['WindGustDir'].fillna(global_mode)

# Filling Columns with Mode
categorical_columns = ['WindDir9am', 'WindDir3pm', 'RainToday']
for col in categorical_columns: 
    df[col] = df.groupby('Location')[col].transform(
        lambda x: x.fillna(x.mode()[0] if not x.mode().empty else x)
)

cloud_columns = ['Cloud9am', 'Cloud3pm']
for col in cloud_columns:
    df[col] = df.groupby('Location')[col].transform(
        lambda x: x.fillna(x.median())
    )
    df[col] = df[col].fillna(df[col].median()) # fallback for locations where all values were NaN (median() returns NaN too)

# Must run after Rainfall is fully imputed above, since RainToday is derived from it
df['RainToday'] = np.where(df['Rainfall'] > 1, 'Yes', 'No')


# Downcasting Datatypes to Float32 for Continuous Value Columns
continuous_columns = df.select_dtypes(include='float').columns
df[continuous_columns] = df[continuous_columns].astype('float32')


# Downcast Clouds to int8
clouds_columns = ['Cloud9am', 'Cloud3pm']

for cloud in clouds_columns:
    df[cloud] = df[cloud].round()  # round first: astype('Int8') truncates instead of rounding
    df[cloud] = df[cloud].astype('Int8')

# Downcast Str to Category
str_columns = df.select_dtypes(include='str').columns
for col in str_columns: 
    df[col] = df[col].astype('category')

# Downcast Rain Tomorrow to Boolean Value
df['RainTomorrow'] = df['RainTomorrow'].astype('bool')

# Save New Dataset
df.to_parquet(OUTPUT_PATH, engine='pyarrow')
print("Done: data_cleaning.py")
print("New Dataset Saved in: data/processed/clean_weather_training_data.csv")
df.isna().sum()


Done: data_cleaning.py
New Dataset Saved in: data/processed/clean_weather_training_data.csv


Location             0
MinTemp              0
MaxTemp              0
Rainfall             0
Evaporation      42501
Sunshine         47287
WindGustDir          0
WindGustSpeed        0
WindDir9am           0
WindDir3pm           0
WindSpeed9am         0
WindSpeed3pm         0
Humidity9am          0
Humidity3pm          0
Pressure9am          0
Pressure3pm          0
Cloud9am             0
Cloud3pm             0
Temp9am              0
Temp3pm              0
RainToday            0
RainTomorrow         0
dtype: int64

In [12]:
df['row ID'].nunique()

KeyError: 'row ID'

In [ ]:
df.drop(columns=['row ID'], inplace=True)

In [ ]:
df.duplicated().sum()

np.int64(30)

In [ ]:
df = df.drop_duplicates()

In [ ]:
for col in df.columns: 
    print(f"Missing Values for {col}: {df[col].isna().sum()}")

Missing Values for Location: 0
Missing Values for MinTemp: 413
Missing Values for MaxTemp: 202
Missing Values for Rainfall: 977
Missing Values for Evaporation: 42501
Missing Values for Sunshine: 47287
Missing Values for WindGustDir: 6491
Missing Values for WindGustSpeed: 6450
Missing Values for WindDir9am: 6976
Missing Values for WindDir3pm: 2618
Missing Values for WindSpeed9am: 905
Missing Values for WindSpeed3pm: 1805
Missing Values for Humidity9am: 1203
Missing Values for Humidity3pm: 2476
Missing Values for Pressure9am: 9718
Missing Values for Pressure3pm: 9706
Missing Values for Cloud9am: 37542
Missing Values for Cloud3pm: 39972
Missing Values for Temp9am: 584
Missing Values for Temp3pm: 1874
Missing Values for RainToday: 977
Missing Values for RainTomorrow: 0


In [ ]:
numerical_columns = df.select_dtypes(include='number').columns
for col in numerical_columns: 
    print(f"Min Value for {col}: {df[col].min()}")
    print(f"Max Value for {col}: {df[col].max()}")

Min Value for MinTemp: -8.5
Max Value for MinTemp: 33.9
Min Value for MaxTemp: -4.1
Max Value for MaxTemp: 48.1
Min Value for Rainfall: 0.0
Max Value for Rainfall: 371.0
Min Value for Evaporation: 0.0
Max Value for Evaporation: 86.2
Min Value for Sunshine: 0.0
Max Value for Sunshine: 14.5
Min Value for WindGustSpeed: 6.0
Max Value for WindGustSpeed: 135.0
Min Value for WindSpeed9am: 0.0
Max Value for WindSpeed9am: 130.0
Min Value for WindSpeed3pm: 0.0
Max Value for WindSpeed3pm: 87.0
Min Value for Humidity9am: 0.0
Max Value for Humidity9am: 100.0
Min Value for Humidity3pm: 0.0
Max Value for Humidity3pm: 100.0
Min Value for Pressure9am: 980.5
Max Value for Pressure9am: 1041.0
Min Value for Pressure3pm: 978.2
Max Value for Pressure3pm: 1039.6
Min Value for Cloud9am: 0.0
Max Value for Cloud9am: 9.0
Min Value for Cloud3pm: 0.0
Max Value for Cloud3pm: 9.0
Min Value for Temp9am: -7.0
Max Value for Temp9am: 40.2
Min Value for Temp3pm: -5.1
Max Value for Temp3pm: 46.7
Min Value for RainTomorro

In [ ]:
continuous_columns = df.select_dtypes(include='float').columns

df[continuous_columns] = df[continuous_columns].astype('float32')
df.info()

<class 'pandas.DataFrame'>
Index: 99486 entries, 0 to 99515
Data columns (total 22 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Location       99486 non-null  str    
 1   MinTemp        99073 non-null  float32
 2   MaxTemp        99284 non-null  float32
 3   Rainfall       98509 non-null  float32
 4   Evaporation    56985 non-null  float32
 5   Sunshine       52199 non-null  float32
 6   WindGustDir    92995 non-null  str    
 7   WindGustSpeed  93036 non-null  float32
 8   WindDir9am     92510 non-null  str    
 9   WindDir3pm     96868 non-null  str    
 10  WindSpeed9am   98581 non-null  float32
 11  WindSpeed3pm   97681 non-null  float32
 12  Humidity9am    98283 non-null  float32
 13  Humidity3pm    97010 non-null  float32
 14  Pressure9am    89768 non-null  float32
 15  Pressure3pm    89780 non-null  float32
 16  Cloud9am       61944 non-null  float32
 17  Cloud3pm       59514 non-null  float32
 18  Temp9am        98902 n

In [ ]:
df[df['MinTemp'] > df['MaxTemp']]

,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,WindDir3pm,WindSpeed9am,WindSpeed3pm,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow


In [ ]:
categorical_columns = df.select_dtypes(include="str").columns
for col in categorical_columns: 
    print(f"Unique Values for {col}: {df[col].nunique()}")

Unique Values for Location: 49
Unique Values for WindGustDir: 16
Unique Values for WindDir9am: 16
Unique Values for WindDir3pm: 16
Unique Values for RainToday: 2


In [ ]:
for col in categorical_columns:  # type: ignore
    print(df[col].unique().tolist())
    print(20 * '-')

['Albury', 'BadgerysCreek', 'Cobar', 'CoffsHarbour', 'Moree', 'Newcastle', 'NorahHead', 'NorfolkIsland', 'Penrith', 'Richmond', 'Sydney', 'SydneyAirport', 'WaggaWagga', 'Williamtown', 'Wollongong', 'Canberra', 'Tuggeranong', 'MountGinini', 'Ballarat', 'Bendigo', 'Sale', 'MelbourneAirport', 'Melbourne', 'Mildura', 'Nhil', 'Portland', 'Watsonia', 'Dartmoor', 'Brisbane', 'Cairns', 'GoldCoast', 'Townsville', 'Adelaide', 'MountGambier', 'Nuriootpa', 'Woomera', 'Albany', 'Witchcliffe', 'PearceRAAF', 'PerthAirport', 'Perth', 'SalmonGums', 'Walpole', 'Hobart', 'Launceston', 'AliceSprings', 'Darwin', 'Katherine', 'Uluru']
--------------------
['W', 'WNW', 'N', 'NNE', 'SW', 'ENE', 'SSE', 'NE', 'WSW', 'NNW', 'S', 'ESE', nan, 'NW', 'E', 'SSW', 'SE']
--------------------
['W', 'NNW', 'ENE', 'SSE', 'S', 'NE', nan, 'SSW', 'N', 'WSW', 'SE', 'ESE', 'E', 'NW', 'NNE', 'SW', 'WNW']
--------------------
['WNW', 'WSW', 'NW', 'W', 'SSE', 'ESE', 'ENE', 'SSW', 'E', 'SW', 'NNW', 'N', 'S', nan, 'SE', 'NNE', 'NE'

In [ ]:
df['Location'].value_counts().sort_index(ascending=True)

Location
Adelaide            2178
Albany              2051
Albury              2142
AliceSprings        2119
BadgerysCreek       2041
Ballarat            2122
Bendigo             2110
Brisbane            2202
Cairns              2101
Canberra            2393
Cobar               2090
CoffsHarbour        2066
Dartmoor            2067
Darwin              2217
GoldCoast           2057
Hobart              2239
Katherine           1065
Launceston          2072
Melbourne           1695
MelbourneAirport    2139
Mildura             2124
Moree               2020
MountGambier        2140
MountGinini         2025
Newcastle           2066
Nhil                1136
NorahHead           2028
NorfolkIsland       2038
Nuriootpa           2110
PearceRAAF          1953
Penrith             2059
Perth               2262
PerthAirport        2167
Portland            2113
Richmond            2060
Sale                2093
SalmonGums          2031
Sydney              2361
SydneyAirport       2100
Townsville      

In [ ]:
numeric_cols = df.select_dtypes(include='number')
outlier_summary = {}

for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    count = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_summary[col] = count

pd.Series(outlier_summary).sort_values(ascending=False)

RainTomorrow     22357
Rainfall         17712
WindGustSpeed     2093
WindSpeed3pm      1717
Evaporation       1367
WindSpeed9am      1203
Humidity9am        978
Pressure9am        885
Pressure3pm        725
Temp3pm            483
MaxTemp            326
Temp9am            209
MinTemp             47
Sunshine             0
Humidity3pm          0
Cloud9am             0
Cloud3pm             0
dtype: int64

In [ ]:
print(df['MinTemp'].min())
print(df['MaxTemp'].max())

-8.5
48.1


In [ ]:
df['MaxTemp'] = df.groupby('Location')['MaxTemp'].transform(lambda x: x.fillna(x.median()))
df['MaxTemp'].isna().sum()

np.int64(0)

In [ ]:
df['MinTemp'] = df.groupby('Location')['MaxTemp'].transform(lambda x: x.fillna(x.median()))
df['MinTemp'].isna().sum()

np.int64(0)

In [ ]:
df["Rainfall"].isna().sum()

np.int64(977)

In [ ]:
df['RainToday'].isna().sum()

np.int64(977)

In [ ]:
df['RainToday'].value_counts()

RainToday
No     76453
Yes    22056
Name: count, dtype: int64

In [ ]:
df.loc[(df['Rainfall'].isna()) & (df['RainToday'] == 'No'), 'Rainfall'] = 0

In [ ]:
df.isna().sum()

Location             0
MinTemp              0
MaxTemp              0
Rainfall           977
Evaporation      42501
Sunshine         47287
WindGustDir       6491
WindGustSpeed     6450
WindDir9am        6976
WindDir3pm        2618
WindSpeed9am       905
WindSpeed3pm      1805
Humidity9am       1203
Humidity3pm       2476
Pressure9am       9718
Pressure3pm       9706
Cloud9am         37542
Cloud3pm         39972
Temp9am            584
Temp3pm           1874
RainToday          977
RainTomorrow         0
dtype: int64

In [ ]:
df['Rainfall'] = df.groupby('Location')['Rainfall'].transform(lambda x: x.fillna(x.median()))

In [ ]:
df['Evaporation'].value_counts()

Evaporation
4.000000     2296
8.000000     1764
2.200000     1453
2.000000     1406
2.400000     1403
             ... 
40.799999       1
43.200001       1
42.799999       1
33.000000       1
39.599998       1
Name: count, Length: 327, dtype: int64

In [ ]:
(df["Evaporation"].isna().sum() / len(df)) * 100

np.float64(42.72058380073578)

In [ ]:
df['Sunshine'].nunique()

145

In [ ]:
df['Sunshine'].value_counts()

Sunshine
0.0     1626
10.7     771
11.0     762
10.8     754
10.5     719
        ... 
14.0      11
14.1       4
14.2       2
14.5       1
14.3       1
Name: count, Length: 145, dtype: int64

In [ ]:
(df['Sunshine'].isna().sum() / len(df)) * 100

np.float64(47.53131093822247)

In [ ]:
df['WindGustDir'].value_counts()

WindGustDir
W      6843
SE     6475
E      6341
SSE    6329
N      6323
SW     6233
S      6228
WSW    6164
SSW    6023
WNW    5664
NW     5599
ENE    5558
ESE    5101
NE     4956
NNW    4589
NNE    4569
Name: count, dtype: int64

In [ ]:
(df['WindGustDir'].isna().sum() / len(df)) * 100

np.float64(6.52453611563436)

In [ ]:
df.groupby('Location')['WindGustDir'].apply(lambda x: x.isna().all())

Location
Adelaide            False
Albany               True
Albury              False
AliceSprings        False
BadgerysCreek       False
Ballarat            False
Bendigo             False
Brisbane            False
Cairns              False
Canberra            False
Cobar               False
CoffsHarbour        False
Dartmoor            False
Darwin              False
GoldCoast           False
Hobart              False
Katherine           False
Launceston          False
Melbourne           False
MelbourneAirport    False
Mildura             False
Moree               False
MountGambier        False
MountGinini         False
Newcastle            True
Nhil                False
NorahHead           False
NorfolkIsland       False
Nuriootpa           False
PearceRAAF          False
Penrith             False
Perth               False
PerthAirport        False
Portland            False
Richmond            False
Sale                False
SalmonGums          False
Sydney              False
Syd

In [ ]:
df['WindGustDir'] = df.groupby('Location')['WindGustDir'].transform(
    lambda x: x.fillna(x.mode()[0] if not x.mode().empty else x)
)

global_mode = df['WindGustDir'].mode()[0]
df['WindGustDir'] = df['WindGustDir'].fillna(global_mode)

In [ ]:
df['WindGustDir'] = df['WindGustDir'].astype('category')

In [ ]:
(df['WindGustSpeed'].isna().sum() / len(df)) * 100

np.float64(6.483324286834328)

In [ ]:
df['WindGustSpeed'].value_counts()

WindGustSpeed
35.0     6353
39.0     6074
31.0     5848
37.0     5563
33.0     5404
         ... 
111.0       1
117.0       1
122.0       1
130.0       1
6.0         1
Name: count, Length: 67, dtype: int64

In [ ]:
print(df['WindGustSpeed'].min())
print(df['WindGustSpeed'].max())

6.0
135.0


In [ ]:
df['WindGustSpeed'] = df.groupby('Location')['WindGustSpeed'].transform(lambda x: x.fillna(df['WindGustSpeed'].median()))
df['WindGustSpeed'].isna().sum()

np.int64(0)

In [ ]:
df['WindGustSpeed'] = df['WindGustSpeed'].astype('float32')

In [ ]:
(df['WindDir9am'].isna().sum() / len(df)) * 100

np.float64(7.012041895342058)

In [ ]:
df['WindDir9am'].value_counts()

WindDir9am
N      8052
E      6333
SE     6311
SSE    6214
S      5995
NW     5975
SW     5808
W      5790
NNE    5600
NNW    5457
ENE    5357
NE     5323
ESE    5312
SSW    5147
WNW    5041
WSW    4795
Name: count, dtype: int64

In [ ]:
df['WindDir9am'] = df.groupby('Location')['WindDir9am'].transform(
    lambda x: x.fillna(x.mode()[0] if not x.mode().empty else x)
)

df['WindDir9am'].isna().sum()

np.int64(0)

In [ ]:

(df['WindDir3pm'].isna().sum() / len(df)) * 100

np.float64(2.6315260438654686)

In [ ]:
df['WindDir3pm'] = df.groupby('Location')['WindDir3pm'].transform(
    lambda x: x.fillna(x.mode()[0] if not x.mode().empty else x)
)

df['WindDir3pm'].isna().sum()

np.int64(0)

In [ ]:
df.isna().sum()

Location             0
MinTemp              0
MaxTemp              0
Rainfall             0
Evaporation      42501
Sunshine         47287
WindGustDir          0
WindGustSpeed        0
WindDir9am           0
WindDir3pm           0
WindSpeed9am       905
WindSpeed3pm      1805
Humidity9am       1203
Humidity3pm       2476
Pressure9am       9718
Pressure3pm       9706
Cloud9am         37542
Cloud3pm         39972
Temp9am            584
Temp3pm           1874
RainToday          977
RainTomorrow         0
dtype: int64

In [ ]:
       
for col in ['WindSpeed9am', 'WindSpeed3pm']:
    df[col] = df.groupby('Location')[col].transform(lambda x: x.fillna(df[col].median()))
    print(df[col].isna().sum())

0
0


In [ ]:
df.isna().sum()

Location             0
MinTemp              0
MaxTemp              0
Rainfall             0
Evaporation      42501
Sunshine         47287
WindGustDir          0
WindGustSpeed        0
WindDir9am           0
WindDir3pm           0
WindSpeed9am         0
WindSpeed3pm         0
Humidity9am       1203
Humidity3pm       2476
Pressure9am       9718
Pressure3pm       9706
Cloud9am         37542
Cloud3pm         39972
Temp9am            584
Temp3pm           1874
RainToday          977
RainTomorrow         0
dtype: int64

In [ ]:
df['Humidity3pm'].nunique()

101

In [ ]:
       
for col in ['Humidity3pm', 'Humidity9am']:
    df[col] = df.groupby('Location')[col].transform(lambda x: x.fillna(df[col].median()))
    print(df[col].isna().sum())

0
0


In [ ]:
df['Pressure3pm'].value_counts()

Pressure3pm
1015.299988    549
1015.700012    545
1013.500000    545
1015.599976    543
1015.500000    536
              ... 
990.500000       1
1039.599976      1
986.799988       1
1038.400024      1
989.500000       1
Name: count, Length: 536, dtype: int64

In [ ]:
for col in ['Pressure9am', 'Pressure3pm']:
    df[col] = df.groupby('Location')[col].transform(lambda x: x.fillna(df[col].median()))
    print(df[col].isna().sum())

0
0


In [ ]:
df.isna().sum()

Location             0
MinTemp              0
MaxTemp              0
Rainfall             0
Evaporation      42501
Sunshine         47287
WindGustDir          0
WindGustSpeed        0
WindDir9am           0
WindDir3pm           0
WindSpeed9am         0
WindSpeed3pm         0
Humidity9am          0
Humidity3pm          0
Pressure9am          0
Pressure3pm          0
Cloud9am         37542
Cloud3pm         39972
Temp9am            584
Temp3pm           1874
RainToday          977
RainTomorrow         0
dtype: int64

In [ ]:
df['Cloud3pm'].value_counts()

Cloud3pm
7.0    12759
1.0    10294
8.0     8677
6.0     6213
2.0     5005
5.0     4745
3.0     4734
4.0     3676
0.0     3410
9.0        1
Name: count, dtype: int64

In [ ]:
df['Cloud3pm'].isna().sum() / len(df)

np.float64(0.40178517580363066)

In [ ]:
df.loc[df["Cloud3pm"] == 9]

,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,WindDir3pm,WindSpeed9am,WindSpeed3pm,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow
73149,Woomera,24.6,24.6,0.2,11.6,11.4,SSE,39.0,SE,ESE,26.0,17.0,45.0,13.0,1019.0,1015.200012,6.0,9.0,14.8,23.700001,No,0


In [ ]:
df.loc[df['Cloud3pm'] == 9.0, 'Cloud3pm'] = np.nan

In [ ]:
df.isna().sum()

Location             0
MinTemp              0
MaxTemp              0
Rainfall             0
Evaporation      42501
Sunshine         47287
WindGustDir          0
WindGustSpeed        0
WindDir9am           0
WindDir3pm           0
WindSpeed9am         0
WindSpeed3pm         0
Humidity9am          0
Humidity3pm          0
Pressure9am          0
Pressure3pm          0
Cloud9am         37542
Cloud3pm         39973
Temp9am            584
Temp3pm           1874
RainToday          977
RainTomorrow         0
dtype: int64

In [ ]:
for col in ['Temp9am', 'Temp3pm']: 
    print((df[col].isna().sum() / len(df)) * 100)

0.5870172687614338
1.8836821261282994


In [ ]:
df['Temp3pm'].value_counts()

Temp3pm
 18.500000    609
 18.400000    605
 19.200001    594
 20.000000    590
 17.799999    587
             ... 
-3.900000       1
-2.400000       1
-4.200000       1
-3.800000       1
 43.799999      1
Name: count, Length: 491, dtype: int64

In [ ]:
df.info()

<class 'pandas.DataFrame'>
Index: 99486 entries, 0 to 99515
Data columns (total 22 columns):
 #   Column         Non-Null Count  Dtype   
---  ------         --------------  -----   
 0   Location       99486 non-null  str     
 1   MinTemp        99486 non-null  float32 
 2   MaxTemp        99486 non-null  float32 
 3   Rainfall       99486 non-null  float32 
 4   Evaporation    56985 non-null  float32 
 5   Sunshine       52199 non-null  float32 
 6   WindGustDir    99486 non-null  category
 7   WindGustSpeed  99486 non-null  float32 
 8   WindDir9am     99486 non-null  str     
 9   WindDir3pm     99486 non-null  str     
 10  WindSpeed9am   99486 non-null  float32 
 11  WindSpeed3pm   99486 non-null  float32 
 12  Humidity9am    99486 non-null  float32 
 13  Humidity3pm    99486 non-null  float32 
 14  Pressure9am    99486 non-null  float32 
 15  Pressure3pm    99486 non-null  float32 
 16  Cloud9am       61944 non-null  float32 
 17  Cloud3pm       59513 non-null  float32 
 18  Te

In [ ]:
for col in ['Temp9am', 'Temp3pm']: 
    df[col] = df[col].astype('float32')

In [ ]:

for col in ['Temp9am', 'Temp3pm']: 
    df[col] = df.groupby('Location')[col].transform(lambda x: x.fillna(df[col].median()))
    print(df[col].isna().sum())

0
0


In [ ]:
df.isna().sum()

Location             0
MinTemp              0
MaxTemp              0
Rainfall             0
Evaporation      42501
Sunshine         47287
WindGustDir          0
WindGustSpeed        0
WindDir9am           0
WindDir3pm           0
WindSpeed9am         0
WindSpeed3pm         0
Humidity9am          0
Humidity3pm          0
Pressure9am          0
Pressure3pm          0
Cloud9am         37542
Cloud3pm         39973
Temp9am              0
Temp3pm              0
RainToday          977
RainTomorrow         0
dtype: int64

In [ ]:
df['RainToday'].value_counts()

RainToday
No     76453
Yes    22056
Name: count, dtype: int64

In [ ]:
(df['RainToday'].isna().sum() / len(df)) * 100

np.float64(0.9820477253080835)

In [ ]:
df['RainToday'] = df.groupby('Location')['RainToday'].transform(
    lambda x: x.fillna(x.mode()[0] if not x.mode().empty else x)
)
(df['RainToday'].isna().sum() / len(df)) * 100

np.float64(0.0)

In [ ]:
df.isna().sum()

Location             0
MinTemp              0
MaxTemp              0
Rainfall             0
Evaporation      42501
Sunshine         47287
WindGustDir          0
WindGustSpeed        0
WindDir9am           0
WindDir3pm           0
WindSpeed9am         0
WindSpeed3pm         0
Humidity9am          0
Humidity3pm          0
Pressure9am          0
Pressure3pm          0
Cloud9am         37542
Cloud3pm         39973
Temp9am              0
Temp3pm              0
RainToday            0
RainTomorrow         0
dtype: int64